# Pobreza e informalidad laboral en el Perú
### Integración de la ENAHO 2024 (Módulo 500 — Empleo e Ingresos) con el Mapa de Pobreza Distrital (INEI)

## 1. Problema que se está trabajando
El MTPE dispone de amplios microdatos de la ENAHO, pero no cuenta con un sistema exploratorio que identifique
de forma automática perfiles de trabajadores o territorios vulnerables al combinar simultáneamente
informalidad, subocupación, ingresos y pobreza. Hoy esa información vive en fuentes separadas que no se
cruzan de forma sistemática.

**Objetivo:** identificar patrones de informalidad laboral y pobreza a nivel distrital que no son evidentes
al mirar cada fuente por separado.

## 2. Fuentes de datos que se integran

| Fuente | Institución | Año de referencia | Nivel de agregación | Enlace |
|---|---|---|---|---|
| ENAHO 2024 — Módulo 500 (Empleo e Ingresos) | INEI | 2024 | Persona (agregada a distrito en este notebook) |http://webinei.inei.gob.pe/anda_inei/index.php/catalog/816/get_microdata |
| Mapa de Pobreza Monetaria Distrital | INEI | 2018 | Distrito | https://www.gob.pe/institucion/inei/informes-publicaciones/3204872-mapa-de-pobreza-provincial-y-distrital-2018|

**⚠️ Nota metodológica:** hay una diferencia de 6 años entre ambas fuentes (ENAHO 2024 vs. Mapa de Pobreza 2018),
porque INEI no ha publicado una actualización distrital más reciente — el mapa distrital solo se recalcula con
metodología censal cada varios años. El informe anual de pobreza sí llega hasta 2024, pero solo a nivel
departamental/dominio geográfico, no distrital. Este notebook prioriza el detalle distrital y asume que la
pobreza estructural cambia lentamente entre 2018 y 2024; el desfase se documenta como limitación conocida,
no se oculta.

## 3. Unidad de análisis
**Distrito**, identificado por `UBIGEO` (código de 6 dígitos). Los microdatos de persona de la ENAHO se
agregan a nivel distrital (tasas e ingresos ponderados por el factor de expansión) antes de cruzarse con
el Mapa de Pobreza.

## 4. Columnas comunes / llave de integración
`UBIGEO` (texto de 6 dígitos: departamento + provincia + distrito) está presente en ambas fuentes y es la
llave usada en el `merge` del Paso 3.

**Antes de correr este notebook:**
1. Descarga el archivo completo de la ENAHO 2024 Módulo 500 desde `proyectos.inei.gob.pe/microdatos` y
   colócalo en la carpeta `data/`.
2. Ajusta la variable `RUTA_ENAHO` en la celda de configuración con el nombre real de tu archivo.
3. Descarga el Anexo Estadístico del Mapa de Pobreza Distrital (INEI) y colócalo como `Anexo Estadístico.xlsx`.


In [20]:
# Librerías
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)

## Configuración

Cambia estas rutas según donde guardes tus archivos.


In [25]:
# ==== CONFIGURACIÓN — AJUSTA ESTO ====
RUTA_ENAHO = "data/2024Empleo e Ingresos.csv"   # archivo completo descargado de INEI
# Si tu archivo viene separado por tabulaciones (típico al copiar desde INEI/SPSS exportado),
# usa sep="\t". Si es CSV normal, usa sep=",". Verifica abriendo las primeras líneas si tienes dudas.
SEPARADOR = ","


## Paso 1 — Cargar solo las columnas relevantes de la ENAHO

El archivo completo tiene más de 700 columnas y puede pesar varios cientos de MB.
Como en Pandas no necesitas todo, usamos `usecols` para traer solo lo que vas a usar — esto reduce
drásticamente el uso de memoria y el tiempo de carga.


In [26]:
# Columnas relevantes para pobreza + informalidad laboral
# (ver diccionario de variables ya armado para el detalle de cada una)
COLUMNAS_ENAHO = [
    # Identificación y geografía
    "AÑO",         # Año de la encuesta
    "MES",         # Mes de la encuesta
    "UBIGEO",      # Código de ubicación geográfica (6 dígitos: departamento+provincia+distrito)
    "DOMINIO",     # Dominio geográfico (1=Costa Norte...8=Lima Metropolitana)
    "ESTRATO",     # Estrato socioeconómico/tamaño del centro poblado del conglomerado
    "CONGLOME",    # Conglomerado (unidad muestral)
    "VIVIENDA",    # Número de vivienda
    "HOGAR",       # Número de hogar dentro de la vivienda
    "CODPERSO",    # Código de la persona dentro del hogar
    # Informalidad (clave)
    "CODINFOR",    # Condición de informalidad del trabajador, calculada por INEI
    # Perfil del trabajador
    "P207",        # Sexo
    "P208A",       # Edad en años cumplidos
    "P301A",       # Nivel educativo alcanzado
    # Ocupación y sector
    "P505R4",   # Ocupación principal — código y clasificación CNO-2015
    "P506R4",   # Actividad económica del empleador — código y clasificación CIIU rev. 4 (el "sector")
    "P507",     # Categoría ocupacional (empleador, independiente, empleado, obrero, TFNR, trab. del hogar)
    # Formalidad del vínculo
    "P510A1",   # Formalidad tributaria del empleador/negocio (RUC / persona jurídica registrada)
    "P511A",    # Tipo de contrato (indefinido, plazo fijo, CAS, locación de servicios, sin contrato)
    "P512A",    # Tamaño de la empresa (número de trabajadores)
    # Horas y subempleo
    "I513T",    # Horas trabajadas en la ocupación principal (versión imputada por INEI)
    "P518",     # Horas trabajadas en ocupación(es) secundaria(s)
    "P520",     # Horas normales semanales declaradas en todas las ocupaciones
    "P521",     # Indicador de subempleo visible: ¿quería trabajar más horas?
    "P521A",    # Complemento de P521: ¿estaba disponible para trabajar más horas?
    # Ingresos (versión imputada, más limpia)
    "I524A1",   # Ingreso en la ocupación principal, versión imputada (usar esta, no la cruda)
    "D524A1",   # Ingreso en la ocupación principal, versión derivada por INEI
    # Seguridad social
    "P558T",    # Afiliación a algún seguro de salud (SIS, EsSalud, EPS, privado, etc.) — resumen/total
    # Condición de ocupación y factor de expansión
    "OCU500",   # Condición de ocupación resumida (ocupado, desocupado, inactivo)
    "FAC500A",  # Factor de expansión muestral — representación de la población
]

enaho = pd.read_csv(
    RUTA_ENAHO,
    sep=SEPARADOR,
    usecols=lambda c: c in COLUMNAS_ENAHO,
    encoding="latin-1",     # los microdatos de INEI suelen venir en latin-1
    low_memory=False,
)

#Pasamos las columnas a nombres reconocibles

RENOMBRAR_COLUMNAS = {
    # Perfil del trabajador
    "P207": "sexo",
    "P208A": "edad",
    "P301A": "nivel_educativo",

    # Ocupación y sector
    "P505R4": "ocupacion_principal_cno",
    "P506R4": "actividad_economica_ciiu",
    "P507": "categoria_ocupacional",

    # Formalidad del vínculo
    "P510A1": "formalidad_tributaria_empleador",
    "P511A": "tipo_contrato",
    "P512A": "tamaño_empresa",

    # Horas y subempleo
    "I513T": "horas_semanales_ocup_principal",
    "P518": "horas_semanales_ocup_secundaria",
    "P520": "horas_normales_todas_ocupaciones",
    "P521": "quiere_trabajar_mas_horas",
    "P521A": "disponible_trabajar_mas_horas",

    # Ingresos (versión imputada, más limpia)
    "I524A1": "ingreso_principal_imputado",
    "D524A1": "ingreso_principal_derivado",

    # Seguridad social
    "P558T": "afiliado_seguro_salud",

    # Condición de ocupación y factor de expansión
    "OCU500": "condicion_ocupacion",
    "FAC500A": "factor_expansion",
}

enaho = enaho.rename(columns=RENOMBRAR_COLUMNAS)

print(f"Filas: {enaho.shape[0]:,} | Columnas: {enaho.shape[1]}")
enaho.head(10)


Filas: 85,992 | Columnas: 29


,AÑO,MES,CONGLOME,VIVIENDA,HOGAR,CODPERSO,UBIGEO,DOMINIO,ESTRATO,CODINFOR,ocupacion_principal_cno,actividad_economica_ciiu,categoria_ocupacional,formalidad_tributaria_empleador,tipo_contrato,tamaño_empresa,horas_semanales_ocup_secundaria,horas_normales_todas_ocupaciones,quiere_trabajar_mas_horas,disponible_trabajar_mas_horas,afiliado_seguro_salud,sexo,edad,nivel_educativo,ingreso_principal_derivado,horas_semanales_ocup_principal,ingreso_principal_imputado,condicion_ocupacion,factor_expansion
0,2024,1,15009,13,11,1,10101,4,4,1,,,,,,,,,,,600,1,45,11,,,,4,136.036972
1,2024,1,15009,13,11,2,10101,4,4,2,7129,9000,2,3,,1,12,,2,,200,2,19,9,,35,,1,75.679695
2,2024,1,15009,47,11,1,10101,4,4,1,4224,8010,3,1,1,4,14,,2,,,2,39,8,11267,36,11267,1,68.032333
3,2024,1,15009,47,11,2,10101,4,4,2,5212,4771,3,2,7,1,,,2,,150,2,19,9,6447,36,6447,1,75.679695
4,2024,1,15009,59,11,1,10101,4,4,1,8322,4922,2,3,,1,,,2,,,1,57,5,,54,,1,81.480087
5,2024,1,15009,71,11,1,10101,4,4,1,3314,6010,1,2,,1,28,,2,,,1,53,8,,32,,1,161.209213
6,2024,1,15009,71,11,2,10101,4,4,2,5312,8521,3,,1,5,,,2,,,2,45,8,18255,40,18255,1,99.098305
7,2024,1,15009,84,11,1,10101,4,4,1,3411,8411,3,,5,5,,,2,,200,1,30,10,26811,54,26811,1,167.057617
8,2024,1,15009,84,11,2,10101,4,4,2,5212,4721,2,3,,1,,,2,,7,2,27,6,,72,,1,39.068672
9,2024,1,15009,106,11,1,10101,4,4,1,8322,4922,2,3,,1,,,2,,,1,41,8,,36,,1,91.023361


## Paso 1d — Tabla resumen de la Fuente 1 (ENAHO)

Antes de seguir, revisamos la forma, los tipos de dato y los estadísticos básicos del dataset cargado.


In [27]:
print("Forma del dataset:", enaho.shape)
print()
print("Tipos de dato por columna:")
print(enaho.dtypes)
print()
print("Estadísticos descriptivos (columnas numéricas):")
enaho.describe()


Forma del dataset: (85992, 29)

Tipos de dato por columna:
AÑO                                   int64
MES                                   int64
CONGLOME                              int64
VIVIENDA                              int64
HOGAR                                 int64
CODPERSO                              int64
UBIGEO                                int64
DOMINIO                               int64
ESTRATO                               int64
CODINFOR                              int64
ocupacion_principal_cno                 str
actividad_economica_ciiu                str
categoria_ocupacional                   str
formalidad_tributaria_empleador         str
tipo_contrato                           str
tamaño_empresa                          str
horas_semanales_ocup_secundaria         str
horas_normales_todas_ocupaciones        str
quiere_trabajar_mas_horas               str
disponible_trabajar_mas_horas           str
afiliado_seguro_salud                   str
sexo             

,AÑO,MES,CONGLOME,VIVIENDA,HOGAR,CODPERSO,UBIGEO,DOMINIO,ESTRATO,CODINFOR,sexo,edad,condicion_ocupacion,factor_expansion
count,85992.0,85992.000000,85992.000000,85992.000000,85992.000000,85992.000000,85992.000000,85992.000000,85992.000000,85992.000000,85992.000000,85992.000000,85992.00000,85992.000000
mean,2024.0,6.463136,17775.083391,76.982940,11.119639,2.144281,130677.258001,4.876872,4.065390,1.978917,1.521258,42.941878,1.88314,313.328510
std,0.0,3.452580,1633.012611,67.566902,1.240747,1.335164,67324.584182,2.400233,2.431031,1.251793,0.499551,19.591572,1.34252,337.838775
min,2024.0,1.000000,10651.000000,1.000000,11.000000,1.000000,10101.000000,1.000000,1.000000,0.000000,1.000000,14.000000,0.00000,0.458707
25%,2024.0,3.000000,16382.000000,31.000000,11.000000,1.000000,70107.000000,3.000000,2.000000,1.000000,1.000000,26.000000,1.00000,103.922873
50%,2024.0,6.000000,17741.000000,66.000000,11.000000,2.000000,140106.000000,5.000000,4.000000,2.000000,2.000000,42.000000,1.00000,219.439072
75%,2024.0,9.000000,19193.000000,104.000000,11.000000,3.000000,180301.000000,7.000000,7.000000,2.000000,2.000000,58.000000,4.00000,385.663361
max,2024.0,12.000000,21001.000000,945.000000,55.000000,16.000000,250401.000000,8.000000,8.000000,16.000000,2.000000,98.000000,4.00000,2707.721924


## Paso 1b — Limpieza y derivación de variables

- Filtrar solo personas ocupadas (para que las tasas de informalidad tengan sentido).
- Confirmar la dirección de `CODINFOR` (¡importante! revisa esto contra el diccionario oficial antes de interpretar resultados).


In [28]:
# UBIGEO como texto de 6 dígitos (a veces se pierde el cero inicial al leer como número)
enaho["UBIGEO"] = enaho["UBIGEO"].astype(str).str.zfill(6)

# Filtrar solo ocupados (OCU500 == 1 es "ocupado")
ocupados = enaho[enaho["condicion_ocupacion"] == 1].copy()

# Verificar CODINFOR
ocupados["ES_INFORMAL"] = (ocupados["CODINFOR"] == 1).astype(int)

print(f"Personas ocupadas en la muestra: {len(ocupados):,}")
ocupados[["UBIGEO", "CODINFOR", "ES_INFORMAL"]].head()


Personas ocupadas en la muestra: 58,579


,UBIGEO,CODINFOR,ES_INFORMAL
1,010101,2,0
2,010101,1,1
3,010101,2,0
4,010101,1,1
5,010101,1,1


## Paso 1c — Agregar indicadores por distrito (ponderado por el factor de expansión)

Todo cálculo agregado en ENAHO **debe** usar `FAC500A`, o el resultado no es representativo a nivel distrital/nacional.


In [29]:
def promedio_ponderado(df, columna_valor, columna_peso="factor_expansion"):
    d = df[[columna_valor, columna_peso]].dropna()
    if len(d) == 0:
        return np.nan
    return np.average(d[columna_valor], weights=d[columna_peso])

def tasa_ponderada(df, columna_flag, columna_peso="factor_expansion"):
    # % ponderado de personas con columna_flag == 1
    return promedio_ponderado(df, columna_flag, columna_peso) * 100

# Convert relevant columns to numeric, coercing errors to NaN
ocupados["ingreso_principal_imputado"] = pd.to_numeric(ocupados["ingreso_principal_imputado"], errors="coerce")
ocupados["horas_semanales_ocup_principal"] = pd.to_numeric(ocupados["horas_semanales_ocup_principal"], errors="coerce")
ocupados["quiere_trabajar_mas_horas"] = pd.to_numeric(ocupados["quiere_trabajar_mas_horas"], errors="coerce")

filas = []
for ubigeo, grupo in ocupados.groupby("UBIGEO"):
    filas.append({
        "UBIGEO": ubigeo,
        "n_muestra": len(grupo),
        "tasa_informalidad_pct": tasa_ponderada(grupo, "ES_INFORMAL"),
        "ingreso_promedio_soles": promedio_ponderado(grupo, "ingreso_principal_imputado"),
        "horas_promedio_semana": promedio_ponderado(grupo, "horas_semanales_ocup_principal"),
        "tasa_subempleo_horas_pct": tasa_ponderada(
            grupo.assign(SUBEMPLEO=(grupo["quiere_trabajar_mas_horas"] == 1).astype(int)), "SUBEMPLEO"
        ),
    })

resumen_ubigeo = pd.DataFrame(filas).sort_values("tasa_informalidad_pct", ascending=False)
resumen_ubigeo.head()

,UBIGEO,n_muestra,tasa_informalidad_pct,ingreso_promedio_soles,horas_promedio_semana,tasa_subempleo_horas_pct
1239,230406,8,100.0,NaN,49.785391,21.079063
529,090607,7,100.0,2950.000000,53.120925,0.000000
252,040706,6,100.0,21551.795867,32.688143,0.000000
625,110203,8,100.0,11685.836889,36.582130,50.778100
935,151023,8,100.0,17907.295600,55.505686,0.000000


## Paso 2 — Cargar el Mapa de Pobreza Distrital (INEI)

Se usa el Anexo Estadístico del **Mapa de Pobreza Monetaria Distrital 2018** de INEI, que reporta resultados
a nivel de distrito (ver la nota metodológica sobre el desfase de años frente a la ENAHO 2024, explicada al inicio del notebook).


In [30]:
# Plantilla para completar con los valores reales del informe INEI 2024
RUTA_INEI = "data/Anexo Estadístico.xlsx"
pobreza_2024 = pd.read_excel(RUTA_INEI)

pobreza_2024.columns = [
    "UBIGEO",
    "DEPARTAMENTO",
    "PROVINCIA",
    "DISTRITO",
    "POBLACION_PROYECTADA_2020",
    "INTERVALO_CONFIANZA_INFERIOR",
    "INTERVALO_CONFIANZA_SUPERIOR",
    "GRUPOS_ROBUSTOS",
    "UBICACION_POBREZA_MONETARIA_TOTAL"
]

# Eliminar las 5 primeras filas
pobreza_2024 = pobreza_2024.iloc[6:]

# UBIGEO como texto de 6 dígitos (a veces se pierde el cero inicial al leer como número)
pobreza_2024["UBIGEO"] = pobreza_2024["UBIGEO"].astype(str).str.zfill(6)

pobreza_2024.head(10)

c:\Users\tormeno\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'A3.23'!$A:$I.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


,UBIGEO,DEPARTAMENTO,PROVINCIA,DISTRITO,POBLACION_PROYECTADA_2020,INTERVALO_CONFIANZA_INFERIOR,INTERVALO_CONFIANZA_SUPERIOR,GRUPOS_ROBUSTOS,UBICACION_POBREZA_MONETARIA_TOTAL
6,050410,AYACUCHO,HUANTA,UCHURACCAY,3672,73.254961,89.268381,1,1
7,060308,CAJAMARCA,CELENDÍN,OXAMARCA,5824,74.592076,87.404338,1,2
8,060402,CAJAMARCA,CHOTA,ANGUIA,3216,71.738599,85.685853,1,3
9,050411,AYACUCHO,HUANTA,PUCACOLPA,2299,65.84465,90.477852,2,4
10,090722,HUANCAVELICA,TAYACAJA,PICHOS,1894,65.638998,88.611786,2,5
11,061007,CAJAMARCA,SAN MARCOS,JOSÉ SABOGAL,13137,69.782681,84.243027,2,6
12,060202,CAJAMARCA,CAJABAMBA,CACHACHI,25575,70.324367,83.389375,2,7
13,060809,CAJAMARCA,JAEN,SALLIQUE,7509,70.825236,82.859754,2,8
14,060604,CAJAMARCA,CUTERVO,CUJILLO,2569,66.162402,87.095438,2,9
15,060304,CAJAMARCA,CELENDÍN,HUASMIN,10885,68.604769,83.658117,2,10


## Paso 2b — Tabla resumen de la Fuente 2 (Mapa de Pobreza Distrital)


In [31]:
print("Forma del dataset:", pobreza_2024.shape)
print()
print("Tipos de dato por columna:")
print(pobreza_2024.dtypes)
print()
pobreza_2024.describe(include="all")


Forma del dataset: (1915, 9)

Tipos de dato por columna:
UBIGEO                                  str
DEPARTAMENTO                            str
PROVINCIA                               str
DISTRITO                                str
POBLACION_PROYECTADA_2020            object
INTERVALO_CONFIANZA_INFERIOR         object
INTERVALO_CONFIANZA_SUPERIOR         object
GRUPOS_ROBUSTOS                         str
UBICACION_POBREZA_MONETARIA_TOTAL    object
dtype: object



,UBIGEO,DEPARTAMENTO,PROVINCIA,DISTRITO,POBLACION_PROYECTADA_2020,INTERVALO_CONFIANZA_INFERIOR,INTERVALO_CONFIANZA_SUPERIOR,GRUPOS_ROBUSTOS,UBICACION_POBREZA_MONETARIA_TOTAL
count,1874,1873,1873,1873,1873,1873.000000,1873.000000,1873,1915
unique,1874,25,196,1737,1723,1588.000000,1595.000000,28,1875
top,050410,LIMA,LIMA,SANTA ROSA,517,13.874602,24.018398,8,Continúa…
freq,1,170,43,10,4,10.000000,10.000000,197,41


## Paso 3b — Cruce de ambas bases


In [32]:
base_final = resumen_ubigeo.merge(pobreza_2024, on="UBIGEO", how="left")
base_final


,UBIGEO,n_muestra,tasa_informalidad_pct,ingreso_promedio_soles,horas_promedio_semana,tasa_subempleo_horas_pct,DEPARTAMENTO,PROVINCIA,DISTRITO,POBLACION_PROYECTADA_2020,INTERVALO_CONFIANZA_INFERIOR,INTERVALO_CONFIANZA_SUPERIOR,GRUPOS_ROBUSTOS,UBICACION_POBREZA_MONETARIA_TOTAL
0,230406,8,100.000000,NaN,49.785391,21.079063,TACNA,TARATA,SUSAPAYA,488,18.707482,36.310516,12,1176
1,090607,7,100.000000,2950.000000,53.120925,0.000000,HUANCAVELICA,HUAYTARÁ,PILPICHACA,2899,22.083767,45.445849,11,945
2,040706,6,100.000000,21551.795867,32.688143,0.000000,AREQUIPA,ISLAY,PUNTA DE BOMBÓN,6464,8.685132,16.087108,18,1676
3,110203,8,100.000000,11685.836889,36.582130,50.778100,ICA,CHINCHA,CHAVÍN,2658,16.559371,43.603231,12,1083
4,151023,8,100.000000,17907.295600,55.505686,0.000000,LIMA,YAUYOS,PUTINZA,517,16.859525,31.757435,14,1285
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1266,090308,14,16.509265,4787.240617,24.850991,0.000000,HUANCAVELICA,ANGARAES,HUAYLLAY GRANDE,1051,32.255193,63.226357,8,440
1267,060508,6,16.235942,40301.748528,46.100856,0.000000,CAJAMARCA,CONTUMAZÁ,YONÁN,6993,18.812567,33.910639,13,1209
1268,150104,27,13.958620,31314.977645,33.497778,3.255979,LIMA,LIMA,BARRANCO,35915,1.633133,4.711517,23,1837
1269,211204,12,12.469184,6290.350920,23.146397,36.537712,PUNO,SANDIA,PATAMBUCO,4212,36.983766,66.317468,7,313


## Paso 3b — Problemas encontrados en los datos

Antes de analizar los resultados, revisamos valores faltantes, tipos de dato mal detectados y
categorías o valores inconsistentes en ambas fuentes.
*texto en cursiva*

In [33]:
# 1. Valores faltantes en la base de pobreza distrital
print("Valores faltantes por columna:")
print(base_final.isna().sum())
num_columns = base_final.shape[1]
base_final = base_final.dropna(thresh=num_columns - 3)
print()
print("--------Luego de la limpieza--------")
print(base_final.isna().sum())


Valores faltantes por columna:
UBIGEO                                0
n_muestra                             0
tasa_informalidad_pct                 0
ingreso_promedio_soles               67
horas_promedio_semana                 0
tasa_subempleo_horas_pct              0
DEPARTAMENTO                          1
PROVINCIA                             1
DISTRITO                              1
POBLACION_PROYECTADA_2020             1
INTERVALO_CONFIANZA_INFERIOR          1
INTERVALO_CONFIANZA_SUPERIOR          1
GRUPOS_ROBUSTOS                       1
UBICACION_POBREZA_MONETARIA_TOTAL     1
dtype: int64

--------Luego de la limpieza--------
UBIGEO                                0
n_muestra                             0
tasa_informalidad_pct                 0
ingreso_promedio_soles               67
horas_promedio_semana                 0
tasa_subempleo_horas_pct              0
DEPARTAMENTO                          0
PROVINCIA                             0
DISTRITO                              

In [34]:
# 2. Distritos que no encontraron pareja en el cruce por UBIGEO (posible pérdida de información)
sin_pobreza = base_final[base_final["DEPARTAMENTO"].isna()]
print(f"Distritos de ENAHO sin dato de pobreza tras el cruce: {len(sin_pobreza)} de {len(base_final)}")
sin_pobreza[["UBIGEO", "n_muestra", "tasa_informalidad_pct"]].head(10)


Distritos de ENAHO sin dato de pobreza tras el cruce: 0 de 1270


,UBIGEO,n_muestra,tasa_informalidad_pct


In [35]:
# 3. Posibles outliers en ingreso promedio — revisar antes de usar en clustering o regresión
print("Distritos con ingreso promedio inusualmente alto (verificar si son casos reales o error de digitación):")
base_final.sort_values("ingreso_promedio_soles", ascending=False)[
    ["UBIGEO", "n_muestra", "ingreso_promedio_soles"]
].head(10)


Distritos con ingreso promedio inusualmente alto (verificar si son casos reales o error de digitación):


,UBIGEO,n_muestra,ingreso_promedio_soles
160,150122,73,67879.591201
49,150130,53,66636.059594
74,150131,48,66356.767668
1025,140206,4,51251.000000
59,150114,67,50772.647103
414,060310,9,49406.585342
361,150120,42,47078.345718
36,230302,68,45910.308316
351,180106,75,45402.876677
712,040126,41,45368.170921


### <font color=green> 2. Inspección inicial </font>

Revisamos tamaño, tipos de datos y estadísticas descriptivas de cada fuente
(complementa lo ya visto en Paso 1d y Paso 2b).

In [36]:
print("Tamaño de enaho:", enaho.shape)
print("Tamaño de pobreza_2024:", pobreza_2024.shape)

print("\nTipos detectados en enaho:")
display(enaho.dtypes)

print("\nTipos detectados en pobreza_2024:")
display(pobreza_2024.dtypes)


Tamaño de enaho: (85992, 29)
Tamaño de pobreza_2024: (1915, 9)

Tipos detectados en enaho:


AÑO                                   int64
MES                                   int64
CONGLOME                              int64
VIVIENDA                              int64
HOGAR                                 int64
CODPERSO                              int64
UBIGEO                                  str
DOMINIO                               int64
ESTRATO                               int64
CODINFOR                              int64
ocupacion_principal_cno                 str
actividad_economica_ciiu                str
categoria_ocupacional                   str
formalidad_tributaria_empleador         str
tipo_contrato                           str
tamaño_empresa                          str
horas_semanales_ocup_secundaria         str
horas_normales_todas_ocupaciones        str
quiere_trabajar_mas_horas               str
disponible_trabajar_mas_horas           str
afiliado_seguro_salud                   str
sexo                                  int64
edad                            


Tipos detectados en pobreza_2024:


UBIGEO                                  str
DEPARTAMENTO                            str
PROVINCIA                               str
DISTRITO                                str
POBLACION_PROYECTADA_2020            object
INTERVALO_CONFIANZA_INFERIOR         object
INTERVALO_CONFIANZA_SUPERIOR         object
GRUPOS_ROBUSTOS                         str
UBICACION_POBREZA_MONETARIA_TOTAL    object
dtype: object

In [37]:
print("Resumen de enaho:")
display(enaho.describe(include="all").T)

print("Resumen de pobreza_2024:")
display(pobreza_2024.describe(include="all").T)


Resumen de enaho:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
AÑO,85992.0,NaN,NaN,NaN,2024.0,0.0,2024.0,2024.0,2024.0,2024.0,2024.0
MES,85992.0,NaN,NaN,NaN,6.463136,3.45258,1.0,3.0,6.0,9.0,12.0
CONGLOME,85992.0,NaN,NaN,NaN,17775.083391,1633.012611,10651.0,16382.0,17741.0,19193.0,21001.0
VIVIENDA,85992.0,NaN,NaN,NaN,76.98294,67.566902,1.0,31.0,66.0,104.0,945.0
HOGAR,85992.0,NaN,NaN,NaN,11.119639,1.240747,11.0,11.0,11.0,11.0,55.0
CODPERSO,85992.0,NaN,NaN,NaN,2.144281,1.335164,1.0,1.0,2.0,3.0,16.0
UBIGEO,85992,1271,070101,1183,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DOMINIO,85992.0,NaN,NaN,NaN,4.876872,2.400233,1.0,3.0,5.0,7.0,8.0
ESTRATO,85992.0,NaN,NaN,NaN,4.06539,2.431031,1.0,2.0,4.0,7.0,8.0
CODINFOR,85992.0,NaN,NaN,NaN,1.978917,1.251793,0.0,1.0,2.0,2.0,16.0


Resumen de pobreza_2024:


,count,unique,top,freq
UBIGEO,1874,1874,050410,1
DEPARTAMENTO,1873,25,LIMA,170
PROVINCIA,1873,196,LIMA,43
DISTRITO,1873,1737,SANTA ROSA,10
POBLACION_PROYECTADA_2020,1873,1723,517,4
INTERVALO_CONFIANZA_INFERIOR,1873.0,1588.0,13.874602,10.0
INTERVALO_CONFIANZA_SUPERIOR,1873.0,1595.0,24.018398,10.0
GRUPOS_ROBUSTOS,1873,28,8,197
UBICACION_POBREZA_MONETARIA_TOTAL,1915,1875,Continúa…,41


### <font color=green> 3. Valores faltantes </font>

Primero detectamos; todavía no corregimos.

In [38]:
faltantes_enaho = pd.DataFrame({
    "n_faltantes": enaho.isna().sum(),
    "porcentaje": (enaho.isna().mean() * 100).round(2)
})

faltantes_pobreza = pd.DataFrame({
    "n_faltantes": pobreza_2024.isna().sum(),
    "porcentaje": (pobreza_2024.isna().mean() * 100).round(2)
})

print("Faltantes en enaho")
display(faltantes_enaho)

print("Faltantes en pobreza_2024")
display(faltantes_pobreza)


Faltantes en enaho


,n_faltantes,porcentaje
AÑO,0,0.0
MES,0,0.0
CONGLOME,0,0.0
VIVIENDA,0,0.0
HOGAR,0,0.0
CODPERSO,0,0.0
UBIGEO,0,0.0
DOMINIO,0,0.0
ESTRATO,0,0.0
CODINFOR,0,0.0


Faltantes en pobreza_2024


,n_faltantes,porcentaje
UBIGEO,41,2.14
DEPARTAMENTO,42,2.19
PROVINCIA,42,2.19
DISTRITO,42,2.19
POBLACION_PROYECTADA_2020,42,2.19
INTERVALO_CONFIANZA_INFERIOR,42,2.19
INTERVALO_CONFIANZA_SUPERIOR,42,2.19
GRUPOS_ROBUSTOS,42,2.19
UBICACION_POBREZA_MONETARIA_TOTAL,0,0.00


### <font color=green> 4. Categorías y errores de registro </font>

Revisamos valores distintos en variables categóricas. Esto ayuda a encontrar codificaciones
no uniformes y a confirmar la dirección de `CODINFOR` (revisar contra el diccionario oficial).

In [39]:
print("Valores de condicion_ocupacion (OCU500):")
display(enaho["condicion_ocupacion"].value_counts(dropna=False))

print("Valores de CODINFOR:")
display(enaho["CODINFOR"].value_counts(dropna=False))

print("Valores de sexo (P207):")
display(enaho["sexo"].value_counts(dropna=False))

print("Valores de categoria_ocupacional (P507):")
display(enaho["categoria_ocupacional"].value_counts(dropna=False))

print("Valores de GRUPOS_ROBUSTOS (pobreza_2024):")
display(pobreza_2024["GRUPOS_ROBUSTOS"].value_counts(dropna=False))


Valores de condicion_ocupacion (OCU500):


condicion_ocupacion
1    58579
4    23908
2     2173
3     1126
0      206
Name: count, dtype: int64

Valores de CODINFOR:


CODINFOR
1     37872
2     27744
3     11025
4      5327
5      2070
6       927
7       417
0       206
8       204
9       109
10       49
11       21
12       15
13        3
14        2
16        1
Name: count, dtype: int64

Valores de sexo (P207):


sexo
2    44824
1    41168
Name: count, dtype: int64

Valores de categoria_ocupacional (P507):


categoria_ocupacional
2    24637
     23118
3    12505
4    11917
5    10450
1     2106
6      980
7      279
Name: count, dtype: int64

Valores de GRUPOS_ROBUSTOS (pobreza_2024):


GRUPOS_ROBUSTOS
8      197
14     185
10     180
12     158
9      138
6      130
7      129
15     116
11     115
16     105
4       54
19      54
5       47
18      43
NaN     42
17      33
22      32
23      31
20      26
13      24
3       23
2       15
21      13
24       9
26       6
25       4
1        3
28       2
27       1
Name: count, dtype: int64

### <font color=green>  5. Duplicados</font>

Distinguimos duplicados exactos y duplicados por clave. En `enaho` la llave de persona es
`CONGLOME + VIVIENDA + HOGAR + CODPERSO`; en `pobreza_2024` la llave es `UBIGEO`.

In [40]:
LLAVE_PERSONA = ["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"]

print("Filas duplicadas exactas en enaho:", enaho.duplicated().sum())
print("Filas duplicadas exactas en pobreza_2024:", pobreza_2024.duplicated().sum())

print("\nPersonas repetidas en enaho (por llave de persona):")
display(enaho[enaho.duplicated(subset=LLAVE_PERSONA, keep=False)].sort_values(LLAVE_PERSONA))

print("\nUBIGEO repetidos en pobreza_2024:")
display(pobreza_2024[pobreza_2024.duplicated(subset=["UBIGEO"], keep=False)].sort_values("UBIGEO"))


Filas duplicadas exactas en enaho: 0
Filas duplicadas exactas en pobreza_2024: 40

Personas repetidas en enaho (por llave de persona):


,AÑO,MES,CONGLOME,VIVIENDA,HOGAR,CODPERSO,UBIGEO,DOMINIO,ESTRATO,CODINFOR,ocupacion_principal_cno,actividad_economica_ciiu,categoria_ocupacional,formalidad_tributaria_empleador,tipo_contrato,tamaño_empresa,horas_semanales_ocup_secundaria,horas_normales_todas_ocupaciones,quiere_trabajar_mas_horas,disponible_trabajar_mas_horas,afiliado_seguro_salud,sexo,edad,nivel_educativo,ingreso_principal_derivado,horas_semanales_ocup_principal,ingreso_principal_imputado,condicion_ocupacion,factor_expansion



UBIGEO repetidos en pobreza_2024:


,UBIGEO,DEPARTAMENTO,PROVINCIA,DISTRITO,POBLACION_PROYECTADA_2020,INTERVALO_CONFIANZA_INFERIOR,INTERVALO_CONFIANZA_SUPERIOR,GRUPOS_ROBUSTOS,UBICACION_POBREZA_MONETARIA_TOTAL
52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Continúa…
99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Continúa…
146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Continúa…
193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Continúa…
240,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Continúa…
288,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Continúa…
335,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Continúa…
382,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Continúa…
429,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Continúa…
476,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Continúa…


### <font color=green>  6. Conversión de tipos</font>

Convertimos variables que deberían ser numéricas o de texto de largo fijo.

`errors="coerce"` transforma valores inválidos en `NaN`, para poder detectarlos y tratarlos.

In [41]:
enaho["UBIGEO"] = enaho["UBIGEO"].astype(str).str.zfill(6)
pobreza_2024["UBIGEO"] = pobreza_2024["UBIGEO"].astype(str).str.zfill(6)

for col in ["edad", "ingreso_principal_imputado", "ingreso_principal_derivado",
            "horas_semanales_ocup_principal", "horas_semanales_ocup_secundaria",
            "horas_normales_todas_ocupaciones", "quiere_trabajar_mas_horas",
            "disponible_trabajar_mas_horas", "factor_expansion"]:
    enaho[col] = pd.to_numeric(enaho[col], errors="coerce")

for col in ["POBLACION_PROYECTADA_2020", "INTERVALO_CONFIANZA_INFERIOR",
            "INTERVALO_CONFIANZA_SUPERIOR", "UBICACION_POBREZA_MONETARIA_TOTAL"]:
    pobreza_2024[col] = pd.to_numeric(pobreza_2024[col], errors="coerce")

print("Tipos después de convertir:")
display(enaho.dtypes)
display(pobreza_2024.dtypes)


Tipos después de convertir:


AÑO                                   int64
MES                                   int64
CONGLOME                              int64
VIVIENDA                              int64
HOGAR                                 int64
CODPERSO                              int64
UBIGEO                                  str
DOMINIO                               int64
ESTRATO                               int64
CODINFOR                              int64
ocupacion_principal_cno                 str
actividad_economica_ciiu                str
categoria_ocupacional                   str
formalidad_tributaria_empleador         str
tipo_contrato                           str
tamaño_empresa                          str
horas_semanales_ocup_secundaria     float64
horas_normales_todas_ocupaciones    float64
quiere_trabajar_mas_horas           float64
disponible_trabajar_mas_horas       float64
afiliado_seguro_salud                   str
sexo                                  int64
edad                            

UBIGEO                                   str
DEPARTAMENTO                             str
PROVINCIA                                str
DISTRITO                                 str
POBLACION_PROYECTADA_2020            float64
INTERVALO_CONFIANZA_INFERIOR         float64
INTERVALO_CONFIANZA_SUPERIOR         float64
GRUPOS_ROBUSTOS                          str
UBICACION_POBREZA_MONETARIA_TOTAL    float64
dtype: object

### <font color=green> 7. Estandarización de nombres y categorías</font>

Mapeamos códigos numéricos de la ENAHO a etiquetas legibles.
**Importante:** verificar estos mapeos contra el diccionario oficial de la ENAHO 2024 antes
del informe final.

In [42]:
enaho["sexo"] = enaho["sexo"].replace({1: "Hombre", 2: "Mujer"})
enaho["CODINFOR"] = enaho["CODINFOR"].replace({1: "Informal", 2: "Formal"})
enaho["condicion_ocupacion"] = enaho["condicion_ocupacion"].replace({1: "Ocupado", 2: "Desocupado", 3: "Inactivo"})
enaho["quiere_trabajar_mas_horas"] = enaho["quiere_trabajar_mas_horas"].replace({1: "Sí", 2: "No"})
enaho["disponible_trabajar_mas_horas"] = enaho["disponible_trabajar_mas_horas"].replace({1: "Sí", 2: "No"})

for col in ["DEPARTAMENTO", "PROVINCIA", "DISTRITO"]:
    pobreza_2024[col] = pobreza_2024[col].astype("string").str.strip().str.title()

print("Categorías estandarizadas:")
display(enaho["sexo"].value_counts(dropna=False))
display(enaho["CODINFOR"].value_counts(dropna=False))
display(enaho["condicion_ocupacion"].value_counts(dropna=False))


Categorías estandarizadas:


sexo
Mujer     44824
Hombre    41168
Name: count, dtype: int64

CODINFOR
Informal    37872
Formal      27744
3           11025
4            5327
5            2070
6             927
7             417
0             206
8             204
9             109
10             49
11             21
12             15
13              3
14              2
16              1
Name: count, dtype: int64

condicion_ocupacion
Ocupado       58579
4             23908
Desocupado     2173
Inactivo       1126
0               206
Name: count, dtype: int64

### <font color=green> 8. Valores fuera de rango</font>

Detectamos valores imposibles o poco razonables: edad entre 0 y 100 años, jornada semanal
hasta 168 horas, ingresos no negativos.

In [43]:
print("Edades fuera de rango:")
display(enaho[(enaho["edad"] < 0) | (enaho["edad"] > 100)])

print("Horas semanales fuera de rango (ocupación principal):")
display(enaho[(enaho["horas_semanales_ocup_principal"] < 0) | (enaho["horas_semanales_ocup_principal"] > 168)])

print("Ingresos negativos (ocupación principal, imputado):")
display(enaho[enaho["ingreso_principal_imputado"] < 0])

print("Tasas de pobreza fuera de [0, 100]:")
display(pobreza_2024[(pobreza_2024["UBICACION_POBREZA_MONETARIA_TOTAL"] < 0) |
                      (pobreza_2024["UBICACION_POBREZA_MONETARIA_TOTAL"] > 100)])


Edades fuera de rango:


,AÑO,MES,CONGLOME,VIVIENDA,HOGAR,CODPERSO,UBIGEO,DOMINIO,ESTRATO,CODINFOR,ocupacion_principal_cno,actividad_economica_ciiu,categoria_ocupacional,formalidad_tributaria_empleador,tipo_contrato,tamaño_empresa,horas_semanales_ocup_secundaria,horas_normales_todas_ocupaciones,quiere_trabajar_mas_horas,disponible_trabajar_mas_horas,afiliado_seguro_salud,sexo,edad,nivel_educativo,ingreso_principal_derivado,horas_semanales_ocup_principal,ingreso_principal_imputado,condicion_ocupacion,factor_expansion


Horas semanales fuera de rango (ocupación principal):


,AÑO,MES,CONGLOME,VIVIENDA,HOGAR,CODPERSO,UBIGEO,DOMINIO,ESTRATO,CODINFOR,ocupacion_principal_cno,actividad_economica_ciiu,categoria_ocupacional,formalidad_tributaria_empleador,tipo_contrato,tamaño_empresa,horas_semanales_ocup_secundaria,horas_normales_todas_ocupaciones,quiere_trabajar_mas_horas,disponible_trabajar_mas_horas,afiliado_seguro_salud,sexo,edad,nivel_educativo,ingreso_principal_derivado,horas_semanales_ocup_principal,ingreso_principal_imputado,condicion_ocupacion,factor_expansion


Ingresos negativos (ocupación principal, imputado):


,AÑO,MES,CONGLOME,VIVIENDA,HOGAR,CODPERSO,UBIGEO,DOMINIO,ESTRATO,CODINFOR,ocupacion_principal_cno,actividad_economica_ciiu,categoria_ocupacional,formalidad_tributaria_empleador,tipo_contrato,tamaño_empresa,horas_semanales_ocup_secundaria,horas_normales_todas_ocupaciones,quiere_trabajar_mas_horas,disponible_trabajar_mas_horas,afiliado_seguro_salud,sexo,edad,nivel_educativo,ingreso_principal_derivado,horas_semanales_ocup_principal,ingreso_principal_imputado,condicion_ocupacion,factor_expansion


Tasas de pobreza fuera de [0, 100]:


,UBIGEO,DEPARTAMENTO,PROVINCIA,DISTRITO,POBLACION_PROYECTADA_2020,INTERVALO_CONFIANZA_INFERIOR,INTERVALO_CONFIANZA_SUPERIOR,GRUPOS_ROBUSTOS,UBICACION_POBREZA_MONETARIA_TOTAL
108,060417,Cajamarca,Chota,Tacabamba,16668.0,53.996887,69.391249,5,101.0
109,081006,Cusco,Paruro,Omacha,5842.0,47.161615,75.748371,5,102.0
110,200202,Piura,Ayabaca,Frías,21370.0,49.269972,73.463742,5,103.0
111,081205,Cusco,Quispicanchi,Ccatca,15084.0,48.580593,74.120887,5,104.0
112,061302,Cajamarca,Santa Cruz,Andabamba,1311.0,50.977678,71.365092,5,105.0
...,...,...,...,...,...,...,...,...,...
1915,150113,Lima,Lima,Jesús María,81743.0,0.284194,0.754276,26,1870.0
1916,150130,Lima,Lima,San Borja,127102.0,0.238198,0.688218,26,1871.0
1917,040126,Arequipa,Arequipa,Yanahuara,26354.0,0.067831,0.622789,27,1872.0
1918,150122,Lima,Lima,Miraflores,108855.0,0.019503,0.151497,28,1873.0


### <font color=green>  9. Tratamiento básico de valores inválidos y faltantes

Primero convertimos valores inválidos en faltantes. Luego aplicamos tratamientos simples.

**Nota metodológica:** los indicadores agregados por `UBIGEO` (`resumen_ubigeo`, ya calculado
arriba en el Paso 1c) usan promedio *ponderado* por `factor_expansion`, no la mediana simple.
La imputación por mediana aplicada aquí es solo para variables de perfil individual.

In [44]:
enaho.loc[(enaho["edad"] < 0) | (enaho["edad"] > 100), "edad"] = np.nan
enaho.loc[(enaho["horas_semanales_ocup_principal"] < 0) |
          (enaho["horas_semanales_ocup_principal"] > 168), "horas_semanales_ocup_principal"] = np.nan
enaho.loc[enaho["ingreso_principal_imputado"] < 0, "ingreso_principal_imputado"] = np.nan

enaho["edad"] = enaho["edad"].fillna(enaho["edad"].median())
enaho["sexo"] = enaho["sexo"].fillna("No especificado")
enaho["nivel_educativo"] = enaho["nivel_educativo"].fillna("No especificado")

antes = len(enaho)
enaho = enaho[enaho["factor_expansion"].notna()].copy()
print(f"Filas descartadas por no tener factor_expansion: {antes - len(enaho)}")

print("Faltantes después del tratamiento (enaho):")
display(enaho.isna().sum())


Filas descartadas por no tener factor_expansion: 0
Faltantes después del tratamiento (enaho):


AÑO                                     0
MES                                     0
CONGLOME                                0
VIVIENDA                                0
HOGAR                                   0
CODPERSO                                0
UBIGEO                                  0
DOMINIO                                 0
ESTRATO                                 0
CODINFOR                                0
ocupacion_principal_cno                 0
actividad_economica_ciiu                0
categoria_ocupacional                   0
formalidad_tributaria_empleador         0
tipo_contrato                           0
tamaño_empresa                          0
horas_semanales_ocup_secundaria     71625
horas_normales_todas_ocupaciones    80203
quiere_trabajar_mas_horas           23118
disponible_trabajar_mas_horas       78242
afiliado_seguro_salud                   0
sexo                                    0
edad                                    0
nivel_educativo                   

### <font color=green>  10. Eliminación o consolidación de duplicados</font>

Conservamos el primer registro por llave de persona en `enaho`, y por `UBIGEO` en `pobreza_2024`.

In [45]:
enaho_limpio = enaho.drop_duplicates(subset=LLAVE_PERSONA, keep="first")
pobreza_limpio = pobreza_2024.drop_duplicates(subset=["UBIGEO"], keep="first")

print("enaho antes y después:", enaho.shape, enaho_limpio.shape)
print("pobreza_2024 antes y después:", pobreza_2024.shape, pobreza_limpio.shape)


enaho antes y después: (85992, 29) (85992, 29)
pobreza_2024 antes y después: (1915, 9) (1875, 9)


### <font color=green> 11. Integración básica</font>

El cruce ya se hizo arriba en el **Paso 3b** (`base_final`). Aquí lo revisamos con
`how="outer"` para ver cobertura entre fuentes antes de quedarnos con el cruce final.

In [46]:
base_integrada_outer = pd.merge(
    resumen_ubigeo,
    pobreza_limpio,
    on="UBIGEO",
    how="outer",
    indicator=True
)

display(base_integrada_outer)
display(base_integrada_outer["_merge"].value_counts())


,UBIGEO,n_muestra,tasa_informalidad_pct,ingreso_promedio_soles,horas_promedio_semana,tasa_subempleo_horas_pct,DEPARTAMENTO,PROVINCIA,DISTRITO,POBLACION_PROYECTADA_2020,INTERVALO_CONFIANZA_INFERIOR,INTERVALO_CONFIANZA_SUPERIOR,GRUPOS_ROBUSTOS,UBICACION_POBREZA_MONETARIA_TOTAL,_merge
0,010101,245.0,50.977271,22100.993202,42.487500,13.566980,Amazonas,Chachapoyas,Chachapoyas,39051.0,5.773728,12.295522,19,1747.0,both
1,010102,NaN,NaN,NaN,NaN,NaN,Amazonas,Chachapoyas,Asunción,278.0,25.308002,47.731896,10,852.0,right_only
2,010103,NaN,NaN,NaN,NaN,NaN,Amazonas,Chachapoyas,Balsas,1180.0,36.793306,54.672618,8,506.0,right_only
3,010104,15.0,38.745239,7654.977454,36.577750,34.038767,Amazonas,Chachapoyas,Cheto,702.0,27.947204,50.392360,10,754.0,both
4,010105,NaN,NaN,NaN,NaN,NaN,Amazonas,Chachapoyas,Chiliquín,589.0,42.324256,63.767068,6,261.0,right_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1871,250303,42.0,50.316883,12457.497107,33.628648,9.119384,Ucayali,Padre Abad,Curimaná,10219.0,4.422702,13.606388,19,1748.0,both
1872,250304,29.0,52.102318,14268.798758,40.553753,0.000000,Ucayali,Padre Abad,Neshuya,10501.0,7.576868,16.316846,18,1680.0,both
1873,250305,30.0,41.222037,8568.674988,37.978274,5.415431,Ucayali,Padre Abad,Alexander Von Humboldt,6159.0,8.116695,17.682425,18,1657.0,both
1874,250401,20.0,68.413629,23833.105407,34.612537,0.000000,Ucayali,Purús,Purús,3284.0,20.895733,42.960943,11,1012.0,both


_merge
both          1270
right_only     605
left_only        1
Name: count, dtype: int64

### <font color=green>  12. Validación final</font>

Después de limpiar e integrar, volvemos a revisar tipos, faltantes y resumen de `base_final`.

In [47]:
print("Tipos finales:")
display(base_final.dtypes)

print("Faltantes finales:")
display(base_final.isna().sum())

print("Resumen final:")
display(base_final.describe(include="all"))


Tipos finales:


UBIGEO                                   str
n_muestra                              int64
tasa_informalidad_pct                float64
ingreso_promedio_soles               float64
horas_promedio_semana                float64
tasa_subempleo_horas_pct             float64
DEPARTAMENTO                             str
PROVINCIA                                str
DISTRITO                                 str
POBLACION_PROYECTADA_2020             object
INTERVALO_CONFIANZA_INFERIOR          object
INTERVALO_CONFIANZA_SUPERIOR          object
GRUPOS_ROBUSTOS                          str
UBICACION_POBREZA_MONETARIA_TOTAL     object
dtype: object

Faltantes finales:


UBIGEO                                0
n_muestra                             0
tasa_informalidad_pct                 0
ingreso_promedio_soles               67
horas_promedio_semana                 0
tasa_subempleo_horas_pct              0
DEPARTAMENTO                          0
PROVINCIA                             0
DISTRITO                              0
POBLACION_PROYECTADA_2020             0
INTERVALO_CONFIANZA_INFERIOR          0
INTERVALO_CONFIANZA_SUPERIOR          0
GRUPOS_ROBUSTOS                       0
UBICACION_POBREZA_MONETARIA_TOTAL     0
dtype: int64

Resumen final:


,UBIGEO,n_muestra,tasa_informalidad_pct,ingreso_promedio_soles,horas_promedio_semana,tasa_subempleo_horas_pct,DEPARTAMENTO,PROVINCIA,DISTRITO,POBLACION_PROYECTADA_2020,INTERVALO_CONFIANZA_INFERIOR,INTERVALO_CONFIANZA_SUPERIOR,GRUPOS_ROBUSTOS,UBICACION_POBREZA_MONETARIA_TOTAL
count,1270,1270.000000,1270.000000,1203.000000,1270.000000,1270.000000,1270,1270,1270,1270.0,1270.0,1270.000000,1270,1270.0
unique,1270,NaN,NaN,NaN,NaN,NaN,25,196,1203,1238.0,1163.0,1167.000000,28,1270.0
top,230406,NaN,NaN,NaN,NaN,NaN,LIMA,LIMA,SANTA ROSA,1072.0,0.0,15.439507,8,1176.0
freq,1,NaN,NaN,NaN,NaN,NaN,111,41,6,3.0,5.0,4.000000,134,1.0
mean,NaN,46.115748,49.987927,15752.010730,36.500927,13.034083,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,72.536711,13.002484,8149.668758,7.624848,14.744911,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,3.000000,8.070413,480.000000,6.800664,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,14.000000,41.715862,10637.197888,31.593063,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,23.000000,49.106411,15069.730397,36.657563,8.368466,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,44.750000,57.395964,19719.499766,41.729010,19.479116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### <font color=green> 13. Registro de decisiones de limpieza</font>



In [48]:
decisiones = pd.DataFrame([
    ["UBIGEO", "Pérdida del cero inicial al leer como número", "Forzar a texto de 6 dígitos (zfill)", "Es un código, no una cantidad; sin el cero inicial no cruza bien entre fuentes."],
    ["sexo, CODINFOR, condicion_ocupacion, subempleo", "Códigos numéricos poco legibles", "Mapear a etiquetas (Hombre/Mujer, Informal/Formal, etc.)", "Facilita el análisis y evita reinterpretar códigos en cada paso."],
    ["edad", "Valores fuera de [0, 100]", "Convertir a faltante e imputar mediana", "Edades negativas o mayores a 100 años no son razonables en este contexto."],
    ["horas_semanales_ocup_principal", "Valores fuera de [0, 168]", "Convertir a faltante (no se imputa, se usa en promedio ponderado)", "Más de 168 horas en una semana es físicamente imposible."],
    ["ingreso_principal_imputado", "Valores negativos", "Convertir a faltante (no se imputa, se usa en promedio ponderado)", "Un ingreso negativo no tiene sentido; se prefiere excluirlo del promedio ponderado a inventar un valor."],
    ["factor_expansion", "Registros sin factor de expansión", "Eliminar la fila", "Sin este ponderador el registro no puede usarse en ningún cálculo representativo a nivel distrital."],
    ["Llave de persona (CONGLOME+VIVIENDA+HOGAR+CODPERSO)", "Posibles duplicados", "Conservar primer registro", "Decisión simplificada para este laboratorio; en el informe final debe justificarse contra la fuente más confiable."],
    ["UBIGEO (pobreza_2024)", "Filas de cabecera/notas del anexo antes de la tabla", "Descartar primeras 6 filas al leer el Excel", "Esas filas no son datos, son encabezados/metadatos del anexo estadístico del INEI."],
    ["base_final (filas con muchos NaN)", "Filas con más de 3 columnas vacías tras el cruce (visto en Paso 3b)", "Eliminar esas filas (dropna con thresh)", "Un distrito con casi todo vacío no aporta información confiable para el análisis."],
    ["DEPARTAMENTO / distritos sin pareja en el cruce", "UBIGEO de ENAHO sin dato de pobreza (visto en Paso 3b)", "Se mantienen identificados, no se eliminan automáticamente", "El desfase 2018 vs. 2024 puede explicar UBIGEO que cambiaron; se documenta para revisión manual."],
    ["ingreso_promedio_soles", "Posibles outliers (visto en Paso 3b)", "Se identifican y listan, no se eliminan automáticamente", "Un ingreso muy alto puede ser un distrito real o un error de digitación; requiere revisión antes de decidir."],
], columns=["variable", "problema", "decisión", "justificación"])

display(decisiones)


,variable,problema,decisión,justificación
0,UBIGEO,Pérdida del cero inicial al leer como número,Forzar a texto de 6 dígitos (zfill),"Es un código, no una cantidad; sin el cero ini..."
1,"sexo, CODINFOR, condicion_ocupacion, subempleo",Códigos numéricos poco legibles,"Mapear a etiquetas (Hombre/Mujer, Informal/For...",Facilita el análisis y evita reinterpretar cód...
2,edad,"Valores fuera de [0, 100]",Convertir a faltante e imputar mediana,Edades negativas o mayores a 100 años no son r...
3,horas_semanales_ocup_principal,"Valores fuera de [0, 168]","Convertir a faltante (no se imputa, se usa en ...",Más de 168 horas en una semana es físicamente ...
4,ingreso_principal_imputado,Valores negativos,"Convertir a faltante (no se imputa, se usa en ...",Un ingreso negativo no tiene sentido; se prefi...
5,factor_expansion,Registros sin factor de expansión,Eliminar la fila,Sin este ponderador el registro no puede usars...
6,Llave de persona (CONGLOME+VIVIENDA+HOGAR+CODP...,Posibles duplicados,Conservar primer registro,Decisión simplificada para este laboratorio; e...
7,UBIGEO (pobreza_2024),Filas de cabecera/notas del anexo antes de la ...,Descartar primeras 6 filas al leer el Excel,"Esas filas no son datos, son encabezados/metad..."
8,base_final (filas con muchos NaN),Filas con más de 3 columnas vacías tras el cru...,Eliminar esas filas (dropna con thresh),Un distrito con casi todo vacío no aporta info...
9,DEPARTAMENTO / distritos sin pareja en el cruce,UBIGEO de ENAHO sin dato de pobreza (visto en ...,"Se mantienen identificados, no se eliminan aut...",El desfase 2018 vs. 2024 puede explicar UBIGEO...


## Paso 4 — Exportar resultados


In [50]:
base_final.to_csv("output/pobreza_informalidad_departamental_2024.csv", index=False)
print("Archivo guardado")
base_final

Archivo guardado


,UBIGEO,n_muestra,tasa_informalidad_pct,ingreso_promedio_soles,horas_promedio_semana,tasa_subempleo_horas_pct,DEPARTAMENTO,PROVINCIA,DISTRITO,POBLACION_PROYECTADA_2020,INTERVALO_CONFIANZA_INFERIOR,INTERVALO_CONFIANZA_SUPERIOR,GRUPOS_ROBUSTOS,UBICACION_POBREZA_MONETARIA_TOTAL
0,230406,8,100.000000,NaN,49.785391,21.079063,TACNA,TARATA,SUSAPAYA,488,18.707482,36.310516,12,1176
1,090607,7,100.000000,2950.000000,53.120925,0.000000,HUANCAVELICA,HUAYTARÁ,PILPICHACA,2899,22.083767,45.445849,11,945
2,040706,6,100.000000,21551.795867,32.688143,0.000000,AREQUIPA,ISLAY,PUNTA DE BOMBÓN,6464,8.685132,16.087108,18,1676
3,110203,8,100.000000,11685.836889,36.582130,50.778100,ICA,CHINCHA,CHAVÍN,2658,16.559371,43.603231,12,1083
4,151023,8,100.000000,17907.295600,55.505686,0.000000,LIMA,YAUYOS,PUTINZA,517,16.859525,31.757435,14,1285
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1266,090308,14,16.509265,4787.240617,24.850991,0.000000,HUANCAVELICA,ANGARAES,HUAYLLAY GRANDE,1051,32.255193,63.226357,8,440
1267,060508,6,16.235942,40301.748528,46.100856,0.000000,CAJAMARCA,CONTUMAZÁ,YONÁN,6993,18.812567,33.910639,13,1209
1268,150104,27,13.958620,31314.977645,33.497778,3.255979,LIMA,LIMA,BARRANCO,35915,1.633133,4.711517,23,1837
1269,211204,12,12.469184,6290.350920,23.146397,36.537712,PUNO,SANDIA,PATAMBUCO,4212,36.983766,66.317468,7,313


## Diccionario de datos

### Fuente 1: ENAHO 2024 - Módulo 500 (Empleo e Ingresos)

| Variable | Nombre original | Descripción | Tipo | Unidad de medida / valores |
|---|---|---|---|---|
| AÑO | AÑO | Año de la encuesta | Numérica (entero) | 2024 |
| MES | MES | Mes de la encuesta | Numérica (entero) | 1-12 |
| UBIGEO | UBIGEO | Código de ubicación geográfica | Categórica (texto, 6 dígitos) | Departamento+Provincia+Distrito |
| DOMINIO | DOMINIO | Dominio geográfico | Categórica | 1=Costa Norte ... 8=Lima Metropolitana |
| ESTRATO | ESTRATO | Estrato socioeconómico/tamaño del centro poblado | Categórica | Códigos INEI |
| CONGLOME | CONGLOME | Conglomerado (unidad muestral) | Identificador | - |
| VIVIENDA | VIVIENDA | Número de vivienda | Identificador | - |
| HOGAR | HOGAR | Número de hogar dentro de la vivienda | Identificador | - |
| CODPERSO | CODPERSO | Código de la persona dentro del hogar | Identificador | - |
| CODINFOR | CODINFOR | Condición de informalidad del trabajador | Categórica | 1=Informal, 2=Formal (verificar contra diccionario oficial) |
| sexo | P207 | Sexo del entrevistado | Categórica | 1=Hombre, 2=Mujer |
| edad | P208A | Edad en años cumplidos | Numérica (entero) | Años |
| nivel_educativo | P301A | Nivel educativo alcanzado | Categórica (ordinal) | Códigos INEI |
| ocupacion_principal_cno | P505R4 | Ocupación principal (CNO-2015) | Categórica | Código CNO |
| actividad_economica_ciiu | P506R4 | Actividad económica del empleador (CIIU) | Categórica | Código CIIU |
| categoria_ocupacional | P507 | Categoría ocupacional | Categórica | Empleador, independiente, empleado, etc. |
| formalidad_tributaria_empleador | P510A1 | Formalidad tributaria del empleador/negocio | Categórica | RUC / persona natural, etc. |
| tipo_contrato | P511A | Tipo de contrato | Categórica | Indefinido, plazo fijo, CAS, locación, etc. |
| tamaño_empresa | P512A | Tamaño de la empresa | Categórica/Numérica | N° de trabajadores (rangos) |
| horas_semanales_ocup_principal | I513T | Horas trabajadas en ocupación principal (imputada) | Numérica (continua) | Horas/semana |
| horas_semanales_ocup_secundaria | P518 | Horas trabajadas en ocupación(es) secundaria(s) | Numérica (continua) | Horas/semana |
| horas_normales_todas_ocupaciones | P520 | Horas normales semanales declaradas (todas las ocupaciones) | Numérica (continua) | Horas/semana |
| quiere_trabajar_mas_horas | P521 | Indicador de subempleo visible | Categórica (binaria) | 1=Sí, 2=No |
| disponible_trabajar_mas_horas | P521A | Disponibilidad para trabajar más horas | Categórica (binaria) | 1=Sí, 2=No |
| ingreso_principal_imputado | I524A1 | Ingreso en ocupación principal (versión imputada) | Numérica (continua) | Soles (S/) |
| ingreso_principal_derivado | D524A1 | Ingreso en ocupación principal (derivada del hogar) | Numérica (continua) | Soles (S/) |
| afiliado_seguro_salud | P558T | Afiliación a algún seguro de salud | Categórica | SIS, EsSalud, EPS, ninguno, etc. |
| condicion_ocupacion | OCU500 | Condición de ocupación resumida | Categórica | 1=Ocupado, 2=Desocupado, etc. |
| factor_expansion | FAC500A | Factor de expansión muestral | Numérica (continua) | Ponderador poblacional |

### Variables derivadas (Paso 1b y 1c)

| Variable | Descripción | Tipo | Unidad / valores |
|---|---|---|---|
| ES_INFORMAL | Indicador binario derivado de CODINFOR==1 | Categórica (binaria) | 1=Informal, 0=Formal |
| n_muestra | N° de observaciones (personas ocupadas) por UBIGEO | Numérica (entero) | Conteo |
| tasa_informalidad_pct | % ponderado de trabajadores informales por UBIGEO | Numérica (continua) | % (0-100) |
| ingreso_promedio_soles | Ingreso promedio ponderado por UBIGEO | Numérica (continua) | Soles (S/) |
| horas_promedio_semana | Horas promedio ponderadas trabajadas por UBIGEO | Numérica (continua) | Horas/semana |
| tasa_subempleo_horas_pct | % ponderado de personas en subempleo por horas | Numérica (continua) | % (0-100) |

### Fuente 2: Anexo Estadístico — Mapa de Pobreza Monetaria Distrital 2018 (INEI)

| Variable | Descripción | Tipo | Unidad / valores |
|---|---|---|---|
| UBIGEO | Código de ubicación geográfica (llave de integración) | Categórica (texto, 6 dígitos) | Departamento+Provincia+Distrito |
| DEPARTAMENTO | Nombre del departamento | Categórica | Texto |
| PROVINCIA | Nombre de la provincia | Categórica | Texto |
| DISTRITO | Nombre del distrito | Categórica | Texto |
| POBLACION_PROYECTADA_2020 | Población proyectada al 2020 | Numérica (entero) | Habitantes |
| INTERVALO_CONFIANZA_INFERIOR | Límite inferior del intervalo de confianza de pobreza | Numérica (continua) | % |
| INTERVALO_CONFIANZA_SUPERIOR | Límite superior del intervalo de confianza de pobreza | Numérica (continua) | % |
| GRUPOS_ROBUSTOS | Agrupación robusta de niveles de pobreza | Categórica | Códigos/grupos INEI |
| UBICACION_POBREZA_MONETARIA_TOTAL | Nivel de pobreza monetaria total estimado | Numérica (continua) | % |

### Llave de integración
**UBIGEO** (código de 6 dígitos) — usada para unir el resumen agregado de ENAHO por distrito/UBIGEO con el anexo de pobreza monetaria del INEI.
